# 03 classical models

SARIMA, ETS and Prophet forecasting a month ahead (2015-06-30 to 2015-07-31), scored against the baselines from notebook 02. same split and same metrics (RMSE, MAE, WAPE).

built on store 1 first to check the pipeline, then scaled to a sample of stores.

## Set-Up

### Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import sklearn

sys.path.append('../src')
from load_clean_data import load_clean_data
from helpers import evaluate_forecast

In [2]:
df = load_clean_data()
df = df.sort_values('Date')
print(df.shape)

c:\Users\corbi\Documents\Projects\sales-forecasting\sales-forecasting-project\notebooks\../src\load_clean_data.py:6: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv('../data/train.csv')


(1017209, 18)


### Train Test Split

same split as notebook 02. train is everything before 2015-06-30, test is the 32 days from 2015-06-30.

In [3]:
train = df[df['Date'] < '2015-06-30']
test  = df[df['Date'] >= '2015-06-30']

length_check = len(train) + len(test) == len(df)

print(f"Length check: {length_check}")

Length check: True


## Single Store Models (Store 1)

### Extract Store 1 Data

In [4]:
store_1_train = train[train['Store'] == 1]
store_1_test  = test[test['Store'] == 1]

print(f"Store 1 train shape: {store_1_train.shape}")
print(f"Store 1 test shape: {store_1_test.shape}")

Store 1 train shape: (910, 18)
Store 1 test shape: (32, 18)


the models need a single series of sales with the date as the index. asfreq('D') sets the daily frequency, and the null checks confirm store 1 has no missing days.

In [5]:
store_1_train_series = store_1_train.set_index('Date')['Sales']
store_1_test_series  = store_1_test.set_index('Date')['Sales']

store_1_train_series = store_1_train_series.asfreq('D')
store_1_test_series = store_1_test_series.asfreq('D')

print(store_1_train_series.index)
print(store_1_test_series.index)
print(store_1_train_series.isnull().sum())
print(store_1_test_series.isnull().sum())

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
               '2013-01-09', '2013-01-10',
               ...
               '2015-06-20', '2015-06-21', '2015-06-22', '2015-06-23',
               '2015-06-24', '2015-06-25', '2015-06-26', '2015-06-27',
               '2015-06-28', '2015-06-29'],
              dtype='datetime64[us]', name='Date', length=910, freq='D')
DatetimeIndex(['2015-06-30', '2015-07-01', '2015-07-02', '2015-07-03',
               '2015-07-04', '2015-07-05', '2015-07-06', '2015-07-07',
               '2015-07-08', '2015-07-09', '2015-07-10', '2015-07-11',
               '2015-07-12', '2015-07-13', '2015-07-14', '2015-07-15',
               '2015-07-16', '2015-07-17', '2015-07-18', '2015-07-19',
               '2015-07-20', '2015-07-21', '2015-07-22', '2015-07-23',
               '2015-07-24', '2015-07-25', '2015-07-26', '2015-07-27',
               '2015-07-28', '2015-07-29

### SARIMA

order (1, 0, 1) and seasonal order (1, 1, 1, 7). d is 0 because store 1 has no trend, and 7 is the weekly cycle. forecast is clipped at 0 because sales can't be negative. first guess settings, not tuned.

#### Import

In [6]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

#### Model

In [7]:
sarima_model = SARIMAX(store_1_train_series, 
                       order=(1, 0, 1), 
                       seasonal_order=(1, 1, 1, 7))

sarima_results = sarima_model.fit()

In [8]:
sarima_forecast = sarima_results.forecast(steps=len(store_1_test_series)) 

sarima_forecast = sarima_forecast.clip(lower=0)

length_check = len(sarima_forecast) == len(store_1_test_series)
print(f"Length check: {length_check}")

Length check: True


In [9]:
sarima_col = sarima_forecast.rename('SARIMA').rename_axis('Date').reset_index()

store_1_test = store_1_test.merge(sarima_col, on='Date', how='left')

print(store_1_test.shape)
print()
print(store_1_test.isnull().sum())
print()
print(store_1_test[['Store', 'DayOfWeek', 'Date', 'Sales', 'SARIMA']].head(10))

(32, 19)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
SARIMA                       0
dtype: int64

   Store  DayOfWeek       Date  Sales       SARIMA
0      1          2 2015-06-30   5735  4609.887757
1      1          3 2015-07-01   5223  4435.338061
2      1          4 2015-07-02   5558  4041.689974
3      1          5 2015-07-03   4665  4515.724585
4      1          6 2015-07-04   4797  4954.005078
5      1          7 2015-07-05      0     0.000000
6      1          1 2015-07-06

#### Eval

In [10]:
RMSE, MAE, WAPE = evaluate_forecast(store_1_test_series, sarima_forecast)

RMSE: 701.03
MAE: 565.68
WAPE: 14.25%


### ETS 

additive seasonality with period 7 and no trend. additive because sunday is always 0, and multiplicative seasonality can't handle zeros.

#### Import

In [11]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

#### Model

In [12]:
ETS_model = ExponentialSmoothing(store_1_train_series,
                                 trend = None,
                                 seasonal = 'add',
                                 seasonal_periods = 7)

ETS_results = ETS_model.fit()

In [13]:
ETS_forecast = ETS_results.forecast(steps=len(store_1_test_series)) #type: ignore

ETS_forecast = ETS_forecast.clip(lower=0)

length_check = len(ETS_forecast) == len(store_1_test_series)
print(f"Length check: {length_check}")

Length check: True


In [14]:
ets_col = ETS_forecast.rename('ETS').rename_axis('Date').reset_index()

store_1_test = store_1_test.merge(ets_col, on='Date', how='left')

print(store_1_test.shape)
print()
print(store_1_test.isnull().sum())
print()
print(store_1_test[['Store', 'DayOfWeek', 'Date', 'Sales', 'ETS']].head(10))

(32, 20)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
SARIMA                       0
ETS                          0
dtype: int64

   Store  DayOfWeek       Date  Sales          ETS
0      1          2 2015-06-30   5735  4212.551914
1      1          3 2015-07-01   5223  3990.242714
2      1          4 2015-07-02   5558  3645.118021
3      1          5 2015-07-03   4665  4137.169930
4      1          6 2015-07-04   4797  4556.925597
5      1          7 2015-07-05      0     0.000000

#### Eval

In [15]:
RMSE, MAE, WAPE= evaluate_forecast(store_1_test_series, ETS_forecast)

RMSE: 793.03
MAE: 608.60
WAPE: 15.33%


### Prophet

prophet takes a table with columns ds (date) and y (sales), and the dates are a normal column, not the index. default settings, forecast clipped at 0.

#### Import

In [16]:
from prophet import Prophet

Importing plotly failed. Interactive plots will not work.


#### Making the ds / y table

In [17]:
prophet_ds_y_table = store_1_train[['Date', 'Sales']].rename(columns={'Date': 'ds', 'Sales': 'y'})

print(prophet_ds_y_table.head())
print(prophet_ds_y_table.shape[0])

                ds     y
1016095 2013-01-01     0
1014980 2013-01-02  5530
1013865 2013-01-03  4327
1012750 2013-01-04  4486
1011635 2013-01-05  4997
910


#### Model

In [18]:
prophet_model = Prophet()

prophet_model_fitted = prophet_model.fit(prophet_ds_y_table)

17:06:02 - cmdstanpy - INFO - Chain [1] start processing
17:06:02 - cmdstanpy - INFO - Chain [1] done processing


In [19]:
prophet_future_df = prophet_model_fitted.make_future_dataframe(periods=len(store_1_test_series), freq='D')

print(prophet_future_df.head())
print(prophet_future_df.tail())
print(prophet_future_df.shape[0])

          ds
0 2013-01-01
1 2013-01-02
2 2013-01-03
3 2013-01-04
4 2013-01-05
            ds
937 2015-07-27
938 2015-07-28
939 2015-07-29
940 2015-07-30
941 2015-07-31
942


In [20]:
prophet_preds = prophet_model_fitted.predict(prophet_future_df)

prophet_preds = prophet_preds.tail(len(store_1_test_series))

prophet_forecast = prophet_preds.set_index('ds')['yhat'].clip(lower=0)

In [21]:
prophet_col = prophet_forecast.rename('Prophet').rename_axis('Date').reset_index()

store_1_test = store_1_test.merge(prophet_col, on='Date', how='left')

print(store_1_test.shape)
print()
print(store_1_test.isnull().sum())
print()
print(store_1_test[['Store', 'DayOfWeek', 'Date', 'Sales', 'Prophet']].head(10))

(32, 21)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
SARIMA                       0
ETS                          0
Prophet                      0
dtype: int64

   Store  DayOfWeek       Date  Sales      Prophet
0      1          2 2015-06-30   5735  4219.709178
1      1          3 2015-07-01   5223  4030.816088
2      1          4 2015-07-02   5558  3662.776560
3      1          5 2015-07-03   4665  4117.079554
4      1          6 2015-07-04   4797  4578.022782
5      1          7

#### Eval

In [22]:
RMSE, MAE, WAPE = evaluate_forecast(store_1_test_series, prophet_forecast)

RMSE: 766.94
MAE: 607.52
WAPE: 15.31%


### Single Store Model Scores:

| Model | RMSE | MAE | WAPE |
| --- | --- | --- | --- |
| SARIMA | 701.03 | 565.68 | 14.25% |
| ETS | 793.03 | 608.60 | 15.33% |
| Prophet | 766.94 | 607.52 | 15.31% |

### Export and Compare Single Store Scores

In [23]:
Models = ['SARIMA', 'ETS', 'Prophet']
rows   = []

for model in Models:
    print(f"Evaluating model: {model}")
    RMSE, MAE, WAPE = evaluate_forecast(store_1_test['Sales'], store_1_test[model])
    rows.append({ "Model": model, "RMSE": RMSE, "MAE": MAE, "WAPE": WAPE })
    print()

results_df = pd.DataFrame(rows)
results_df.to_csv('../results/single_store_classical_model_results.csv', index=False)

Evaluating model: SARIMA
RMSE: 701.03
MAE: 565.68
WAPE: 14.25%

Evaluating model: ETS
RMSE: 793.03
MAE: 608.60
WAPE: 15.33%

Evaluating model: Prophet
RMSE: 766.94
MAE: 607.52
WAPE: 15.31%



baseline rows come from notebook 02 (single_store_baseline_model_results.csv).

In [25]:
baseline_scores = pd.read_csv('../results/single_store_baseline_model_results.csv')
classical_scores = pd.read_csv('../results/single_store_classical_model_results.csv')

concatenated_scores = pd.concat([baseline_scores, classical_scores]).sort_values(by='WAPE', ascending=True)

concatenated_scores.style.hide(axis='index').format(precision=2)

Model,RMSE,MAE,WAPE
SARIMA,701.03,565.68,14.25
FullSeasonalAverage,702.58,567.18,14.29
Prophet,766.94,607.52,15.31
ETS,793.03,608.60,15.33
SeasonalAverage28Day,910.90,704.87,17.76
NaiveForecast,1029.52,797.59,20.10


| Model | RMSE | MAE | WAPE |
| --- | --- | --- | --- |
| SARIMA | 701.03 | 565.68 | 14.25% |
| Full Seasonal Average | 702.58 |567.18 | 14.29% |
| Prophet | 766.94 | 607.52 | 15.31% |
| ETS | 793.03 | 608.60 | 15.33% |
| 28 Day Seasonal Average | 910.90 | 704.87 | 17.76% |
| Naive | 1029.52 | 797.59 | 20.10% |

Finding: SARIMA (14.25%) and the full seasonal average (14.29%) are effectively tied. prophet and ETS are about a point behind (15.31%, 15.33%). the 28 day average and naive are clearly worse. the weekly pattern explains most of what is predictable here.

caveats:
- one store over one month, so small gaps could flip on another store or month
- none of the models are tuned
- none of the models see Promo